# NeuroGolf 2026: Task-Level ONNX Bundle

A validated, task-level submission for the **2026 NeuroGolf Championship**.

Every one of the 400 ARC-AGI tasks is solved by its own minimal ONNX graph, scored under
`max(1, 25 - ln(params + activation_bytes))` per task. This notebook loads the per-task nets
and assembles the competition `submission.zip`.

**Public LB ~ 7186.7**


## 1. Setup

The per-task ONNX nets are attached as a Kaggle **Dataset** (under `/kaggle/input`), either as
loose `task*.onnx` files or inside a `.zip`. We collect them by task name.

In [ ]:
import zipfile, glob, os

onnx = glob.glob('/kaggle/input/**/task*.onnx', recursive=True)
if not onnx:
    os.makedirs('/kaggle/working/nets', exist_ok=True)
    for zp in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        with zipfile.ZipFile(zp) as z:
            z.extractall('/kaggle/working/nets')
    onnx = glob.glob('/kaggle/working/nets/**/task*.onnx', recursive=True)

seen, files = set(), []
for f in sorted(onnx, key=lambda p: os.path.basename(p)):
    b = os.path.basename(f)
    if b not in seen:
        seen.add(b)
        files.append(f)

print(f'found {len(files)} task nets')
assert len(files) == 400, f'expected 400 nets, got {len(files)}'

## 2. Assemble the submission

Zip the 400 nets (flat, `task###.onnx` at the archive root) into `/kaggle/working/submission.zip`.

In [ ]:
out = '/kaggle/working/submission.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=os.path.basename(f))

print(f'wrote {out}: {os.path.getsize(out)/1024:.0f} KB, {len(files)} nets')

## 3. Sanity check

Confirm the archive holds 400 uniquely-named task nets.

In [ ]:
with zipfile.ZipFile(out) as z:
    names = z.namelist()

assert len(names) == 400 and len(set(names)) == 400, 'expected 400 unique nets'
print('submission.zip OK -', len(names), 'nets')
print('first:', names[:3])
print('last :', names[-3:])

Submit `submission.zip` from this notebook's output.